# 🦾 Controlling Real Robots with AI Agents

The `Robot` class bridges two worlds:
- **LeRobot's hardware abstraction** — serial buses, cameras, motor drivers
- **Strands Agents** — natural language reasoning and tool calling

The result: an AI agent that can say *"pick up the red cube"* and have a
physical robot actually do it.

> ⚡ This notebook is conceptual — it shows the API and patterns. Running
> the code requires a physical robot (LeRobot-compatible) connected via USB.

## How It Works

The `Robot` class is a Strands `AgentTool`. An agent can call it with:

| Action | What Happens |
|--------|-------------|
| `start` | Launch a task in a background thread — returns immediately |
| `status` | Check progress (IDLE → CONNECTING → RUNNING → COMPLETED) |
| `stop` | Interrupt a running task gracefully |
| `execute` | Blocking version — waits until the task finishes |

## Setting Up a Robot

```python
from strands_robots import Robot

robot = Robot(
    tool_name="orange_arm",         # unique name (used by the agent)
    robot="so101_follower",         # LeRobot robot type
    cameras={
        "front": {"type": "opencv", "index_or_path": "/dev/video0", "fps": 30},
        "wrist": {"type": "opencv", "index_or_path": "/dev/video2", "fps": 30},
    },
    port="/dev/ttyACM0",            # servo serial port
    data_config="so100_dualcam",    # policy config name
    control_frequency=50.0,         # 50Hz = 20ms per action step
    action_horizon=8,               # consume 8 actions per inference call
)
```

The `robot=` parameter accepts:
- A **LeRobot `Robot` instance** — use your existing setup
- A **`RobotConfig` object** — from LeRobot's config system
- A **type string** (e.g., `"so101_follower"`) — auto-builds the config

## Natural Language Robot Control

Give the robot tool to a Strands Agent and it becomes controllable via conversation:

```python
from strands import Agent
from strands_robots import Robot, gr00t_inference

robot = Robot(tool_name="arm", robot="so101_follower", ...)
agent = Agent(tools=[robot, gr00t_inference])

# The agent figures out which tools to call:
agent("Start the arm picking up the red cube using GR00T on port 5555")
# → calls: robot(action="start", instruction="pick up the red cube",
#                policy_provider="groot", policy_port=5555)

agent("How is the arm doing?")
# → calls: robot(action="status")
# → responds: "RUNNING: 127 steps completed, 5.1 seconds elapsed"

agent("Stop the arm, something went wrong")
# → calls: robot(action="stop")
# → responds: "STOPPED after 127 steps"
```

## The Control Loop

Inside, the robot runs this loop in a background thread:

```python
while not shutdown_event.is_set() and elapsed < duration:
    # 1. Read sensors
    obs = self.robot.get_observation()   # cameras + joint positions

    # 2. Ask the policy what to do
    actions = await policy.get_actions(obs, instruction)

    # 3. Execute actions at control frequency
    for action in actions[:action_horizon]:
        self.robot.send_action(action)     # command the motors
        await asyncio.sleep(1 / 50)        # 20ms at 50Hz
        step_count += 1
```

The `control_frequency` determines the timing. At 50Hz, each action step
takes 20ms. With an `action_horizon` of 8, that's 160ms per inference call.

## Task State Machine

```
IDLE → CONNECTING → RUNNING → COMPLETED
                       ↓
                    STOPPED (via stop action)
                       ↓
                    ERROR (if something crashes)
```

The state is tracked in `RobotTaskState` — the agent can check it anytime
with the `status` action.